In [33]:
import pandas as pd
from transformers import pipeline, AutoTokenizer
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import numpy as np

In [34]:
# Load the main dataset and supporting files
data = pd.read_csv("../PII_extraction/raw_data/dataset.csv")
common_names = pd.read_csv("../PII_extraction/utils/common_names.csv")
gender_neutral_job_titles = pd.read_csv("../PII_extraction/raw_data/gender_neutral_job_titles.csv")
# data[["Resume"]].to_csv("../PII_extraction/raw_data/resumes.csv", index=False)
data['Resume_anonymized'] = pd.read_csv("../PII_extraction/anonymized_files/anonymized_text.csv")['anonymized_text']

In [35]:
tokenizer = AutoTokenizer.from_pretrained("ab-ai/pii_model_based_on_distilbert")

def predict_entities_chunked(text, chunk_size=400, overlap=50):
    """Run NER on long text by splitting it into overlapping token chunks.
    Offsets are adjusted so all entities refer to positions in the original text."""
    encoding = tokenizer(text, return_offsets_mapping=True, add_special_tokens=False)
    input_ids = encoding['input_ids']
    offsets = encoding['offset_mapping']

    all_entities = []
    start = 0

    while start < len(input_ids):
        end = min(start + chunk_size, len(input_ids))
        chunk_ids = input_ids[start:end]
        chunk_text = tokenizer.decode(chunk_ids)
        char_offset = offsets[start][0]  # where this chunk starts in the original text

        results = ner_pipeline(chunk_text)
        for entity in results:
            # Shift offsets from chunk position to full text position
            entity['start'] += char_offset
            entity['end'] += char_offset
            all_entities.append(entity)

        # Advance with overlap to avoid missing entities that span chunk boundaries
        start += chunk_size - overlap

    return all_entities

In [43]:
# Load gender from the anonymized target file
data['gender'] = pd.read_csv("../PII_extraction/anonymized_files/anonymized_text_target.csv")['gender']

def run_bow_by_gender(text_col, label):
    for gender in ['M', 'F']:
        mask = data['gender'] == gender
        gender_label = "MEN" if gender == 'M' else "WOMEN"
        subset = data[mask]

        if len(subset) == 0:
            print(f"\nNo data for {gender_label}")
            continue

        vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
        X = vectorizer.fit_transform(subset[text_col].fillna('') + " " + subset['Job_Description'].fillna(''))
        y = subset['decision']

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        model = LogisticRegression(max_iter=1000)
        model.fit(X_train, y_train)

        feature_names = vectorizer.get_feature_names_out()
        coefficients = model.coef_[0]

        print(f"\n=== BoW results: {label} | {gender_label} ===")
        top_positive = np.argsort(coefficients)[-20:]
        print("Words that increase hiring chances:")
        for i in top_positive:
            print(f"  {feature_names[i]:30s} {coefficients[i]:.3f}")

        top_negative = np.argsort(coefficients)[:20]
        print("\nWords that decrease hiring chances:")
        for i in top_negative:
            print(f"  {feature_names[i]:30s} {coefficients[i]:.3f}")

run_bow_by_gender('Resume', "BEFORE anonymization")
run_bow_by_gender('Resume_anonymized', "AFTER anonymization")


=== BoW results: BEFORE anonymization | MEN ===
Words that increase hiring chances:
  hyperparameter tuning          0.290
  hyperparameter                 0.290
  michael                        0.295
  andrew                         0.301
  kevin                          0.302
  robinson                       0.302
  automation                     0.304
  architect                      0.305
  data solutions                 0.313
  excel                          0.315
  jeremy                         0.319
  technical                      0.331
  tuning                         0.334
  dale                           0.351
  marketing                      0.359
  data analyst                   0.377
  ux                             0.436
  gregory                        0.453
  christopher                    0.598
  data                           0.897

Words that decrease hiring chances:
  jeffrey                        -0.582
  mobile                         -0.481
  security        

In [46]:
# Load gender from the anonymized target file
data['gender'] = pd.read_csv("../PII_extraction/anonymized_files/anonymized_text_target.csv")['gender']

def run_bow_by_gender(text_col, label):
    for gender in ['M', 'F']:
        mask = data['gender'] == gender
        gender_label = "MEN" if gender == 'M' else "WOMEN"
        subset = data[mask]

        if len(subset) == 0:
            print(f"\nNo data for {gender_label}")
            continue

        vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
        X = vectorizer.fit_transform(subset[text_col].fillna('') + " " + subset['Job_Description'].fillna(''))
        y = subset['decision']

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        model = LogisticRegression(max_iter=1000)
        model.fit(X_train, y_train)

        feature_names = vectorizer.get_feature_names_out()
        coefficients = model.coef_[0]

        print(f"\n=== BoW results: {label} | {gender_label} ===")
        top_positive = np.argsort(coefficients)[-20:]
        print("Words that increase hiring chances:")
        for i in top_positive:
            print(f"  {feature_names[i]:30s} {coefficients[i]:.3f}")

        top_negative = np.argsort(coefficients)[:20]
        print("\nWords that decrease hiring chances:")
        for i in top_negative:
            print(f"  {feature_names[i]:30s} {coefficients[i]:.3f}")

run_bow_by_gender('Resume', "BEFORE anonymization")
run_bow_by_gender('Resume_anonymized', "AFTER anonymization")


=== BoW results: BEFORE anonymization | MEN ===
Words that increase hiring chances:
  hyperparameter tuning          0.290
  hyperparameter                 0.290
  michael                        0.295
  andrew                         0.301
  kevin                          0.302
  robinson                       0.302
  automation                     0.304
  architect                      0.305
  data solutions                 0.313
  excel                          0.315
  jeremy                         0.319
  technical                      0.331
  tuning                         0.334
  dale                           0.351
  marketing                      0.359
  data analyst                   0.377
  ux                             0.436
  gregory                        0.453
  christopher                    0.598
  data                           0.897

Words that decrease hiring chances:
  jeffrey                        -0.582
  mobile                         -0.481
  security        